# 0) Import libraries 

In [ ]:
!pip install ultralytics

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from sklearn.cluster import KMeans

In [3]:
model = YOLO("/kaggle/input/models/abdelrahmanabr/soccre-player-ball-detection/pytorch/default/1/best.pt")


# 1)Byte tracking Algorithm

In [3]:

# Run tracking on a test match video
results = model.track(
    source="/kaggle/input/datasets/abdelrahmanabr/match-video/Screen Recording 2026-10-13 110113.mp4",  # Replace with your video file path
    conf=0.15,                              # Low confidence threshold helps maintain ball track
    iou=0.5,                                # NMS IoU threshold
    imgsz=1024,                             # Match training resolution
    tracker="bytetrack.yaml",               # Specifies the ByteTracker algorithm
    save=True,                              # Saves the rendered video with IDs
    project="/kaggle/working/runs",
    name="bytetrack_output"
)

print("Tracking complete! Check /kaggle/working/runs/bytetrack_output for the video.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 340ms
Prepared 1 package in 249ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
      

In [5]:

video_path = "/kaggle/input/datasets/abdelrahmanabr/match-video/Screen Recording 2026-10-13 110113.mp4"  # Replace with your video path
cap = cv2.VideoCapture(video_path)

# Video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Output video writer
output_path = "/kaggle/working/runs/team_clustering_output.mp4"
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

def get_upper_torso(frame, box):
    """Extracts the upper body jersey region from a player bounding box."""
    x1, y1, x2, y2 = map(int, box[:4])
    h = y2 - y1
    w = x2 - x1
    
    # Target upper middle torso (jersey area)
    y_start = y1 + int(0.15 * h)
    y_end = y1 + int(0.55 * h)
    x_start = x1 + int(0.2 * w)
    x_end = x2 - int(0.2 * w)
    
    y_start, y_end = max(0, y_start), min(frame.shape[0], y_end)
    x_start, x_end = max(0, x_start), min(frame.shape[1], x_end)
    
    if y_end <= y_start or x_end <= x_start:
        return None
    return frame[y_start:y_end, x_start:x_end]

# --- Step 1: Collect color samples from initial frames to fit K-Means ---
print(" collecting player jersey samples for team color clustering...")
color_samples = []
sample_frames = min(30, total_frames)

for _ in range(sample_frames):
    ret, frame = cap.read()
    if not ret: break
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, cls_id in zip(boxes, classes):
            if model.names[int(cls_id)] == 'player':
                torso = get_upper_torso(frame, box)
                if torso is not None and torso.size > 0:
                    hsv_torso = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
                    mean_color = np.mean(hsv_torso, axis=(0, 1))
                    color_samples.append(mean_color)

cap.release()

# Fit K-Means clustering for 2 teams
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
if len(color_samples) > 10:
    kmeans.fit(np.array(color_samples))
    print("✅ Team color clusters successfully initialized!")
else:
    print(" Warning: Low sample count. Defaulting clusters.")

# --- Step 2: Process entire video with team assignment & tracking ---
print(" Processing video with team clustering and ByteTracker...")
cap = cv2.VideoCapture(video_path)
track_team_cache = {}  # Locks team ID per player to avoid flickering

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, track_id, cls_id in zip(boxes, track_ids, classes):
            class_name = model.names[int(cls_id)]
            x1, y1, x2, y2 = map(int, box[:4])
            
            if class_name == 'player':
                # Check cache or predict team via K-Means
                if track_id in track_team_cache:
                    team = track_team_cache[track_id]
                else:
                    torso = get_upper_torso(frame, box)
                    if torso is not None and torso.size > 0:
                        hsv_torso = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
                        mean_color = np.mean(hsv_torso, axis=(0, 1)).reshape(1, -1)
                        team = kmeans.predict(mean_color)[0]
                        track_team_cache[track_id] = team
                    else:
                        team = 0
                
                # Assign colors & labels: Team 0 = Blue, Team 1 = Red
                color = (255, 0, 0) if team == 0 else (0, 0, 255)
                label = f"Team A #{int(track_id)}" if team == 0 else f"Team B #{int(track_id)}"
                
            elif class_name == 'goalkeeper':
                color = (0, 255, 255)  # Yellow
                label = f"GK #{int(track_id)}"
            elif class_name == 'referee':
                color = (255, 255, 255)  # White
                label = f"Ref #{int(track_id)}"
            else:  # ball
                color = (0, 255, 0)  # Green
                label = "Ball"

            # Draw annotations
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 8), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                        
    out.write(frame)

cap.release()
out.release()
print(f"✅ Processing complete! Saved to {output_path}")

 collecting player jersey samples for team color clustering...
✅ Team color clusters successfully initialized!
 Processing video with team clustering and ByteTracker...
✅ Processing complete! Saved to /kaggle/working/runs/team_clustering_output.mp4


# 2) Byte traker + KNN Clustring for teams 

In [6]:

video_path = "/kaggle/input/datasets/abdelrahmanabr/match-video/classico_analayze.mp4"  # Replace with your video path
cap = cv2.VideoCapture(video_path)

# Video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Output video writer
output_path = "/kaggle/working/runs/team_clustering_output2.mp4"
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

def get_upper_torso(frame, box):
    """Extracts the upper body jersey region from a player bounding box."""
    x1, y1, x2, y2 = map(int, box[:4])
    h = y2 - y1
    w = x2 - x1
    
    # Target upper middle torso (jersey area)
    y_start = y1 + int(0.15 * h)
    y_end = y1 + int(0.55 * h)
    x_start = x1 + int(0.2 * w)
    x_end = x2 - int(0.2 * w)
    
    y_start, y_end = max(0, y_start), min(frame.shape[0], y_end)
    x_start, x_end = max(0, x_start), min(frame.shape[1], x_end)
    
    if y_end <= y_start or x_end <= x_start:
        return None
    return frame[y_start:y_end, x_start:x_end]

# --- Step 1: Collect color samples from initial frames to fit K-Means ---
print(" collecting player jersey samples for team color clustering...")
color_samples = []
sample_frames = min(30, total_frames)

for _ in range(sample_frames):
    ret, frame = cap.read()
    if not ret: break
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, cls_id in zip(boxes, classes):
            if model.names[int(cls_id)] == 'player':
                torso = get_upper_torso(frame, box)
                if torso is not None and torso.size > 0:
                    hsv_torso = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
                    mean_color = np.mean(hsv_torso, axis=(0, 1))
                    color_samples.append(mean_color)

cap.release()

# Fit K-Means clustering for 2 teams
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
if len(color_samples) > 10:
    kmeans.fit(np.array(color_samples))
    print("✅ Team color clusters successfully initialized!")
else:
    print(" Warning: Low sample count. Defaulting clusters.")

# --- Step 2: Process entire video with team assignment & tracking ---
print(" Processing video with team clustering and ByteTracker...")
cap = cv2.VideoCapture(video_path)
track_team_cache = {}  # Locks team ID per player to avoid flickering

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, track_id, cls_id in zip(boxes, track_ids, classes):
            class_name = model.names[int(cls_id)]
            x1, y1, x2, y2 = map(int, box[:4])
            
            if class_name == 'player':
                # Check cache or predict team via K-Means
                if track_id in track_team_cache:
                    team = track_team_cache[track_id]
                else:
                    torso = get_upper_torso(frame, box)
                    if torso is not None and torso.size > 0:
                        hsv_torso = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
                        mean_color = np.mean(hsv_torso, axis=(0, 1)).reshape(1, -1)
                        team = kmeans.predict(mean_color)[0]
                        track_team_cache[track_id] = team
                    else:
                        team = 0
                
                # Assign colors & labels: Team 0 = Blue, Team 1 = Red
                color = (255, 0, 0) if team == 0 else (0, 0, 255)
                label = f"Team A #{int(track_id)}" if team == 0 else f"Team B #{int(track_id)}"
                
            elif class_name == 'goalkeeper':
                color = (0, 255, 255)  # Yellow
                label = f"GK #{int(track_id)}"
            elif class_name == 'referee':
                color = (255, 255, 255)  # White
                label = f"Ref #{int(track_id)}"
            else:  # ball
                color = (0, 255, 0)  # Green
                label = "Ball"

            # Draw annotations
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 8), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                        
    out.write(frame)

cap.release()
out.release()
print(f"✅ Processing complete! Saved to {output_path}")

 collecting player jersey samples for team color clustering...
✅ Team color clusters successfully initialized!
 Processing video with team clustering and ByteTracker...
✅ Processing complete! Saved to /kaggle/working/runs/team_clustering_output2.mp4


# 3) Bytetracker + KNN + Ball Position logic

In [5]:
video_path = "/kaggle/input/datasets/abdelrahmanabr/match-video/classico_analayze.mp4"  # Replace with your video path
cap = cv2.VideoCapture(video_path)

# Video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Output video writer
output_path = "/kaggle/working/runs/possession_hud_output2.mp4"
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

def get_upper_torso(frame, box):
    """Extracts the upper body jersey region from a player bounding box."""
    x1, y1, x2, y2 = map(int, box[:4])
    h = y2 - y1
    w = x2 - x1
    
    y_start = y1 + int(0.15 * h)
    y_end = y1 + int(0.55 * h)
    x_start = x1 + int(0.2 * w)
    x_end = x2 - int(0.2 * w)
    
    y_start, y_end = max(0, y_start), min(frame.shape[0], y_end)
    x_start, x_end = max(0, x_start), min(frame.shape[1], x_end)
    
    if y_end <= y_start or x_end <= x_start:
        return None
    return frame[y_start:y_end, x_start:x_end]

# --- Step 1: Collect color samples to fit K-Means ---
print(" Initializing team color clusters...")
color_samples = []
sample_frames = min(30, total_frames)

for _ in range(sample_frames):
    ret, frame = cap.read()
    if not ret: break
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, cls_id in zip(boxes, classes):
            if model.names[int(cls_id)] == 'player':
                torso = get_upper_torso(frame, box)
                if torso is not None and torso.size > 0:
                    hsv_torso = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
                    mean_color = np.mean(hsv_torso, axis=(0, 1))
                    color_samples.append(mean_color)

cap.release()

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
if len(color_samples) > 10:
    kmeans.fit(np.array(color_samples))
    print("✅ Team color clusters initialized successfully!")
else:
    print("⚠️ Warning: Low sample count. Defaulting clusters.")

# --- Step 2: Process Video with Possession Tracking & Top-Right HUD ---
print(" Processing video with live possession HUD...")
cap = cv2.VideoCapture(video_path)
track_team_cache = {}

# Possession counters
team_a_frames = 0
team_b_frames = 0
possession_threshold_px = 75  # Max distance from player to ball to count possession

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    
    ball_pos = None
    players_data = [] # Stores (team_id, player_center_x, player_center_y)
    
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, track_id, cls_id in zip(boxes, track_ids, classes):
            class_name = model.names[int(cls_id)]
            x1, y1, x2, y2 = map(int, box[:4])
            
            if class_name == 'player':
                if track_id in track_team_cache:
                    team = track_team_cache[track_id]
                else:
                    torso = get_upper_torso(frame, box)
                    if torso is not None and torso.size > 0:
                        hsv_torso = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
                        mean_color = np.mean(hsv_torso, axis=(0, 1)).reshape(1, -1)
                        team = kmeans.predict(mean_color)[0]
                        track_team_cache[track_id] = team
                    else:
                        team = 0
                
                # Colors & Labels
                color = (255, 0, 0) if team == 0 else (0, 0, 255) # Team A: Blue, Team B: Red
                label = f"A #{int(track_id)}" if team == 0 else f"B #{int(track_id)}"
                
                # Save player foot/center position for ball proximity check
                center_x = (x1 + x2) / 2
                center_y = y2  # Use bottom of bounding box (feet position)
                players_data.append((team, center_x, center_y))
                
            elif class_name == 'goalkeeper':
                color = (0, 255, 255)  # Yellow
                label = f"GK #{int(track_id)}"
            elif class_name == 'referee':
                color = (255, 255, 255)  # White
                label = f"Ref #{int(track_id)}"
            else:  # ball
                color = (0, 255, 0)  # Green
                label = "Ball"
                ball_pos = ((x1 + x2) / 2, (y1 + y2) / 2)

            # Draw bounding boxes
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 6), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 2)

    # --- Ball Possession Calculation ---
    if ball_pos is not None and len(players_data) > 0:
        bx, by = ball_pos
        min_dist = float('inf')
        possessing_team = None
        
        for team, px, py in players_data:
            dist = np.sqrt((bx - px)**2 + (by - py)**2)
            if dist < min_dist:
                min_dist = dist
                possessing_team = team
                
        # If a player is close enough to the ball, credit possession
        if min_dist < possession_threshold_px:
            if possessing_team == 0:
                team_a_frames += 1
            else:
                team_b_frames += 1

    # Compute live percentages
    total_poss_frames = team_a_frames + team_b_frames
    if total_poss_frames > 0:
        team_a_pct = int((team_a_frames / total_poss_frames) * 100)
        team_b_pct = 100 - team_a_pct
    else:
        team_a_pct, team_b_pct = 50, 50

    # --- Draw Top-Right HUD Overlay ---
    hud_x = width - 280
    hud_y = 30
    hud_w = 260
    hud_h = 75
    
    # Semi-transparent background box for HUD
    overlay = frame.copy()
    cv2.rectangle(overlay, (hud_x, hud_y), (hud_x + hud_w, hud_y + hud_h), (20, 20, 20), -1)
    alpha = 0.6
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
    
    # HUD border
    cv2.rectangle(frame, (hud_x, hud_y), (hud_x + hud_w, hud_y + hud_h), (200, 200, 200), 1)
    
    # HUD Text
    cv2.putText(frame, "BALL POSSESSION", (hud_x + 15, hud_y + 22), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    
    text_team_a = f"Team Aِ1: {team_a_pct}%"
    text_team_b = f"Team B2: {team_b_pct}%"
    
    cv2.putText(frame, text_team_a, (hud_x + 15, hud_y + 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 100, 100), 2, cv2.LINE_AA) # Blue-ish in BGR
    cv2.putText(frame, text_team_b, (hud_x + 140, hud_y + 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (100, 100, 255), 2, cv2.LINE_AA) # Red-ish in BGR

    out.write(frame)

cap.release()
out.release()
print(f"✅ Processing complete! Video with live possession HUD saved to {output_path}")

 Initializing team color clusters...
✅ Team color clusters initialized successfully!
 Processing video with live possession HUD...
✅ Processing complete! Video with live possession HUD saved to /kaggle/working/runs/possession_hud_output2.mp4


In [5]:
video_path = "/kaggle/input/datasets/abdelrahmanabr/match-video/classico_analayze.mp4"
cap = cv2.VideoCapture(video_path)

# Video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Output video writer
output_path = "/kaggle/working/runs/possession_hud_output_fixed.mp4"
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

def get_clean_jersey_color(frame, box):
    """Extracts upper torso and filters out grass (green) and extreme shadows/highlights."""
    x1, y1, x2, y2 = map(int, box[:4])
    h = y2 - y1
    w = x2 - x1
    
    # Target upper middle torso (jersey area)
    y_start = y1 + int(0.15 * h)
    y_end = y1 + int(0.55 * h)
    x_start = x1 + int(0.25 * w)
    x_end = x2 - int(0.25 * w)
    
    y_start, y_end = max(0, y_start), min(frame.shape[0], y_end)
    x_start, x_end = max(0, x_start), min(frame.shape[1], x_end)
    
    if y_end <= y_start or x_end <= x_start:
        return None
        
    torso = frame[y_start:y_end, x_start:x_end]
    hsv_torso = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
    
    # Mask out green grass pixels (Hue roughly between 35 and 85)
    # Also mask out extreme dark/bright pixels to handle shadows better
    h_channel = hsv_torso[:, :, 0]
    s_channel = hsv_torso[:, :, 1]
    v_channel = hsv_torso[:, :, 2]
    
    # Non-grass and valid lighting mask
    valid_mask = (h_channel < 35) | (h_channel > 85)
    valid_mask &= (v_channel > 40) & (v_channel < 240)  # Filter extreme shadows/glare
    
    if np.sum(valid_mask) > 10:
        # Return mean of non-grass, lighting-normalized pixels (Focus more on Hue & Saturation)
        mean_h = np.mean(h_channel[valid_mask])
        mean_s = np.mean(s_channel[valid_mask])
        return np.array([mean_h, mean_s]) # Using Hue and Saturation avoids shadow brightness bias
    
    return None

# --- Step 1: Collect robust color samples ---
print("🔍 Collecting robust jersey color samples (ignoring grass/shadows)...")
color_samples = []
sample_frames = min(50, total_frames)  # Sample more frames for better distribution

for _ in range(sample_frames):
    ret, frame = cap.read()
    if not ret: break
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, cls_id in zip(boxes, classes):
            if model.names[int(cls_id)] == 'player':
                color_feat = get_clean_jersey_color(frame, box)
                if color_feat is not None:
                    color_samples.append(color_feat)

cap.release()

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
if len(color_samples) > 15:
    kmeans.fit(np.array(color_samples))
    print("✅ Team color clusters successfully re-initialized with shadow-robust filtering!")
else:
    print("⚠️ Warning: Low sample count. Defaulting clusters.")

# --- Step 2: Process Video ---
print("🚀 Processing video with corrected clustering & vertical HUD...")
cap = cv2.VideoCapture(video_path)
track_team_cache = {}

team_a_frames = 0
team_b_frames = 0
possession_threshold_px = 75

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    
    ball_pos = None
    players_data = []
    
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, track_id, cls_id in zip(boxes, track_ids, classes):
            class_name = model.names[int(cls_id)]
            x1, y1, x2, y2 = map(int, box[:4])
            
            if class_name == 'player':
                if track_id in track_team_cache:
                    team = track_team_cache[track_id]
                else:
                    color_feat = get_clean_jersey_color(frame, box)
                    if color_feat is not None:
                        team = kmeans.predict(color_feat.reshape(1, -1))[0]
                        track_team_cache[track_id] = team
                    else:
                        team = 0
                
                # Colors & Labels (Team A: Blue, Team B: Red)
                color = (255, 0, 0) if team == 0 else (0, 0, 255)
                label = f"A #{int(track_id)}" if team == 0 else f"B #{int(track_id)}"
                
                center_x = (x1 + x2) / 2
                center_y = y2
                players_data.append((team, center_x, center_y))
                
            elif class_name == 'goalkeeper':
                color = (0, 255, 255)  # Yellow
                label = f"GK #{int(track_id)}"
            elif class_name == 'referee':
                color = (255, 255, 255)  # White
                label = f"Ref #{int(track_id)}"
            else:  # ball
                color = (0, 255, 0)  # Green
                label = "Ball"
                ball_pos = ((x1 + x2) / 2, (y1 + y2) / 2)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 6), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 2)

    # Ball Possession Calculation
    if ball_pos is not None and len(players_data) > 0:
        bx, by = ball_pos
        min_dist = float('inf')
        possessing_team = None
        
        for team, px, py in players_data:
            dist = np.sqrt((bx - px)**2 + (by - py)**2)
            if dist < min_dist:
                min_dist = dist
                possessing_team = team
                
        if min_dist < possession_threshold_px:
            if possessing_team == 0:
                team_a_frames += 1
            else:
                team_b_frames += 1

    total_poss_frames = team_a_frames + team_b_frames
    if total_poss_frames > 0:
        team_a_pct = int((team_a_frames / total_poss_frames) * 100)
        team_b_pct = 100 - team_a_pct
    else:
        team_a_pct, team_b_pct = 50, 50

    # --- Draw Vertical Top-Right HUD Overlay ---
    hud_w = 200
    hud_h = 80
    hud_x = width - hud_w - 20
    hud_y = 30
    
    overlay = frame.copy()
    cv2.rectangle(overlay, (hud_x, hud_y), (hud_x + hud_w, hud_y + hud_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.rectangle(frame, (hud_x, hud_y), (hud_x + hud_w, hud_y + hud_h), (200, 200, 200), 1)
    
    # HUD Title
    cv2.putText(frame, "BALL POSSESSION", (hud_x + 12, hud_y + 20), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    
    # Stacked Vertically: Team A on top, Team B below
    cv2.putText(frame, f"Team A: {team_a_pct}%", (hud_x + 12, hud_y + 48), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 100, 100), 2, cv2.LINE_AA)
    cv2.putText(frame, f"Team B: {team_b_pct}%", (hud_x + 12, hud_y + 70), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100, 100, 255), 2, cv2.LINE_AA)

    out.write(frame)

cap.release()
out.release()
print(f"✅ Fixed processing complete! Saved to {output_path}")

🔍 Collecting robust jersey color samples (ignoring grass/shadows)...
✅ Team color clusters successfully re-initialized with shadow-robust filtering!
🚀 Processing video with corrected clustering & vertical HUD...
✅ Fixed processing complete! Saved to /kaggle/working/runs/possession_hud_output_fixed.mp4


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from sklearn.cluster import KMeans

# Load your trained model weights
model = YOLO("/kaggle/working/runs/Version3/weights/best.pt")

video_path = "/kaggle/input/datasets/abdelrahmanabr/match-video/classico_analayze.mp4"
cap = cv2.VideoCapture(video_path)

# Video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Output video writer
output_path = "/kaggle/working/runs/possession_hud_output_fixed2.mp4"
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

def get_clean_jersey_color(frame, box):
    """Extracts upper torso and filters out grass (green) and extreme shadows/highlights."""
    x1, y1, x2, y2 = map(int, box[:4])
    h = y2 - y1
    w = x2 - x1
    
    # Target upper middle torso (jersey area)
    y_start = y1 + int(0.15 * h)
    y_end = y1 + int(0.55 * h)
    x_start = x1 + int(0.25 * w)
    x_end = x2 - int(0.25 * w)
    
    y_start, y_end = max(0, y_start), min(frame.shape[0], y_end)
    x_start, x_end = max(0, x_start), min(frame.shape[1], x_end)
    
    if y_end <= y_start or x_end <= x_start:
        return None
        
    torso = frame[y_start:y_end, x_start:x_end]
    hsv_torso = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
    
    # Mask out green grass pixels (Hue roughly between 35 and 85)
    # Also mask out extreme dark/bright pixels to handle shadows better
    h_channel = hsv_torso[:, :, 0]
    s_channel = hsv_torso[:, :, 1]
    v_channel = hsv_torso[:, :, 2]
    
    # Non-grass and valid lighting mask
    valid_mask = (h_channel < 35) | (h_channel > 85)
    valid_mask &= (v_channel > 40) & (v_channel < 240)  # Filter extreme shadows/glare
    
    if np.sum(valid_mask) > 10:
        # Return mean of non-grass, lighting-normalized pixels (Focus more on Hue & Saturation)
        mean_h = np.mean(h_channel[valid_mask])
        mean_s = np.mean(s_channel[valid_mask])
        return np.array([mean_h, mean_s]) # Using Hue and Saturation avoids shadow brightness bias
    
    return None

# --- Step 1: Collect robust color samples ---
print("🔍 Collecting robust jersey color samples (ignoring grass/shadows)...")
color_samples = []
sample_frames = min(50, total_frames)  # Sample more frames for better distribution

for _ in range(sample_frames):
    ret, frame = cap.read()
    if not ret: break
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, cls_id in zip(boxes, classes):
            if model.names[int(cls_id)] == 'player':
                color_feat = get_clean_jersey_color(frame, box)
                if color_feat is not None:
                    color_samples.append(color_feat)

cap.release()

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
if len(color_samples) > 15:
    kmeans.fit(np.array(color_samples))
    print("✅ Team color clusters successfully re-initialized with shadow-robust filtering!")
else:
    print("⚠️ Warning: Low sample count. Defaulting clusters.")

# --- Step 2: Process Video ---
print("🚀 Processing video with corrected clustering & vertical HUD...")
cap = cv2.VideoCapture(video_path)
track_team_cache = {}

team_a_frames = 0
team_b_frames = 0
possession_threshold_px = 75

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    
    ball_pos = None
    players_data = []
    
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, track_id, cls_id in zip(boxes, track_ids, classes):
            class_name = model.names[int(cls_id)]
            x1, y1, x2, y2 = map(int, box[:4])
            
            if class_name == 'player':
                if track_id in track_team_cache:
                    team = track_team_cache[track_id]
                else:
                    color_feat = get_clean_jersey_color(frame, box)
                    if color_feat is not None:
                        team = kmeans.predict(color_feat.reshape(1, -1))[0]
                        track_team_cache[track_id] = team
                    else:
                        team = 0
                
                # Colors & Labels (Team A: Blue, Team B: Red)
                color = (255, 0, 0) if team == 0 else (0, 0, 255)
                label = f"A #{int(track_id)}" if team == 0 else f"B #{int(track_id)}"
                
                center_x = (x1 + x2) / 2
                center_y = y2
                players_data.append((team, center_x, center_y))
                
            elif class_name == 'goalkeeper':
                color = (0, 255, 255)  # Yellow
                label = f"GK #{int(track_id)}"
            elif class_name == 'referee':
                color = (255, 255, 255)  # White
                label = f"Ref #{int(track_id)}"
            else:  # ball
                color = (0, 255, 0)  # Green
                label = "Ball"
                ball_pos = ((x1 + x2) / 2, (y1 + y2) / 2)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 6), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 2)

    # Ball Possession Calculation
    if ball_pos is not None and len(players_data) > 0:
        bx, by = ball_pos
        min_dist = float('inf')
        possessing_team = None
        
        for team, px, py in players_data:
            dist = np.sqrt((bx - px)**2 + (by - py)**2)
            if dist < min_dist:
                min_dist = dist
                possessing_team = team
                
        if min_dist < possession_threshold_px:
            if possessing_team == 0:
                team_a_frames += 1
            else:
                team_b_frames += 1

    total_poss_frames = team_a_frames + team_b_frames
    if total_poss_frames > 0:
        team_a_pct = int((team_a_frames / total_poss_frames) * 100)
        team_b_pct = 100 - team_a_pct
    else:
        team_a_pct, team_b_pct = 50, 50

    # --- Draw Vertical Top-Right HUD Overlay ---
    hud_w = 200
    hud_h = 80
    hud_x = width - hud_w - 20
    hud_y = 30
    
    overlay = frame.copy()
    cv2.rectangle(overlay, (hud_x, hud_y), (hud_x + hud_w, hud_y + hud_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.rectangle(frame, (hud_x, hud_y), (hud_x + hud_w, hud_y + hud_h), (200, 200, 200), 1)
    
    # HUD Title
    cv2.putText(frame, "BALL POSSESSION", (hud_x + 12, hud_y + 20), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    
    # Stacked Vertically: Team A on top, Team B below
    cv2.putText(frame, f"Team A: {team_a_pct}%", (hud_x + 12, hud_y + 48), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 100, 100), 2, cv2.LINE_AA)
    cv2.putText(frame, f"Team B: {team_b_pct}%", (hud_x + 12, hud_y + 70), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100, 100, 255), 2, cv2.LINE_AA)

    out.write(frame)

cap.release()
out.release()
print(f"✅ Fixed processing complete! Saved to {output_path}")